# **Entrenamiento del modelo de ReID**

Entrenamiento del modelo OSNet para reid con los datos en la carpeta ./data creada con obtener_datos_reid.ipynb.

## **Conjunto de datos**

In [ ]:
import torchreid
from torchreid.reid.data import ImageDataset
import sys
import os
import os.path as osp

class SportsMOTReID(ImageDataset):

    def __init__(self, root="", **kwargs):
        self.root = osp.abspath(osp.expanduser(root))
        train_dir = osp.join(self.root, "train")
        query_dir = osp.join(self.root, "query")
        gallery_dir = osp.join(self.root, "gallery")

        train = self._read_split(osp.join(train_dir, "train.txt"))
        query = self._read_split(osp.join(query_dir, "query.txt"))
        gallery = self._read_split(osp.join(gallery_dir, "gallery.txt"))

        super().__init__(train, query, gallery, **kwargs)

    def _read_split(self, fpath):
        data = []
        with open(fpath, "r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                impath, pid, camid = line.split(",")
                pid = int(pid)
                camid = int(camid)
                data.append((impath, pid, camid))
        return data
    
torchreid.data.register_image_dataset("sportsmot_reid", SportsMOTReID)

datamanager = torchreid.data.ImageDataManager(
    root=r".\data",
    sources="sportsmot_reid",
    targets="sportsmot_reid",
    use_gpu=True, # Usar la GPU para ir más rápido
    combineall=False, # No usa query y gallery para entrenar, solo para evaluar
    batch_size_train = 32, # Tamaño de batch
    batch_size_test = 64,
    workers=0,
    height=256, # Tamaño de la imagen
    width=128,
    transforms=["random_flip"], # Se aplica transformación de flip aleatorio para aumentar variabilidad de los datos
    num_instances=4, # Número de instancias por persona en cada batch
    train_sampler="RandomIdentitySampler"
)

from torchreid.reid.models import build_model

model = build_model(
    name='osnet_x1_0', # Uso osnet
    num_classes=datamanager.num_train_pids, # Número de clases = número de personas en train (20 al ser dos partidos)
    loss='triplet', # Usa triplet loss para la pérdida
    pretrained=True, # Usa los pesos preentrenado para ir más rápido y para partir de una buena base sobre la que fine-tunear. Se usa ImageNet
    use_gpu=True # Usar la GPU
).cuda()

import torch
from torchreid.reid.optim import build_optimizer, build_lr_scheduler

optimizer = build_optimizer(model, optim='adam', lr=3e-4) # Valores por defecto
scheduler = build_lr_scheduler(optimizer, lr_scheduler='single_step', stepsize=10) # Usa un scheduler que reduzca el learning rate

from torchreid.reid.engine import ImageTripletEngine 

engine = ImageTripletEngine(
    datamanager,
    model,
    optimizer=optimizer,
    scheduler=scheduler,
    use_gpu=True, # Usar la GPU
    margin=0.3, # Margin para triplet loss. Es el valor por defecto
    weight_t=1.0, # Hace caso al triplet loss
    weight_x=0.0, # No hace caso a otra pérdida que no sea triplet loss
)

## **Entrenamiento**

In [ ]:
engine.run(
    max_epoch=30,               
    save_dir='log/osnet_sportsmot',
    print_freq=2,
    eval_freq=2,
    test_only=False
)

## **Evaluación del entrenamiento**

En este apartado, se evalúa el entrenamiento para escoger el mejor checkpoint del modelo

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt

# Ruta al log
log_path = "log.txt"

with open(log_path, "r", encoding="utf-8") as f:
    text = f.read()

# Extraer métricas
pattern = re.compile(
    r"\*\* Results \*\*\s*"
    r"mAP:\s*([0-9.]+)%\s*"
    r"CMC curve\s*"
    r"Rank-1\s*:\s*([0-9.]+)%\s*"
    r"Rank-5\s*:\s*([0-9.]+)%.*?"
    r'model\.pth\.tar-(\d+)"',
    re.S
)

matches = pattern.findall(text)

# Construir tabla
rows = []
for mAP, rank1, rank5, ckpt in matches:
    eval_id = int(ckpt)
    epoch = eval_id
    rows.append({
        "epoch": epoch,
        "mAP": float(mAP),
        "Rank-1": float(rank1),
        "Rank-5": float(rank5)
    })

df = pd.DataFrame(rows).sort_values("epoch").reset_index(drop=True)

display(df)

# Mejor época es la 12
epoch_escogida = 12
best_idx = df[df["epoch"] == epoch_escogida].index[0]
best_epoch = df.loc[best_idx, "epoch"]
best_map = df.loc[best_idx, "mAP"]

# Gráfica mAP
plt.figure(figsize=(8, 5))
plt.plot(df["epoch"], df["mAP"], marker="o", linewidth=2, label="mAP")
plt.axvline(best_epoch, color="red", linestyle="--", alpha=0.7, label=f"Época escogida ({best_epoch})")
plt.scatter(best_epoch, best_map, color="red", zorder=5)
plt.title("Evolución de mAP por época")
plt.xlabel("Época")
plt.ylabel("mAP (%)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# Gráfica Rank-1 y Rank-5
plt.figure(figsize=(8, 5))
plt.plot(df["epoch"], df["Rank-1"], marker="o", linewidth=2, label="Rank-1")
plt.plot(df["epoch"], df["Rank-5"], marker="s", linewidth=2, label="Rank-5")
plt.axvline(best_epoch, color="red", linestyle="--", alpha=0.7, label=f"Época escogida ({best_epoch})")
plt.title("Evolución de Rank-1 y Rank-5 por época")
plt.xlabel("Época")
plt.ylabel("Accuracy (%)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## **Guardar el modelo en .pt**

In [ ]:
import torch
import torchreid
from pathlib import Path

# Ruta al checkpoint de Torchreid
ckpt_path = Path("log/osnet_sportsmot/model/model.pth.tar-12")
assert ckpt_path.is_file(), f"No existe {ckpt_path}"

# Contruir el mismo modelo que se entrenó
model = torchreid.models.build_model(
name="osnet_x1_0",
num_classes=20,
loss="triplet",
pretrained=False, # No hace falta usar los pesos preentrenados porque se van a sobreescribir con los del checkpoint
use_gpu=True
)

# Cargar pesos desde el checkpoint
state = torch.load(str(ckpt_path), map_location="cuda:0", weights_only=False)
if "state_dict" in state:
    state_dict = state["state_dict"]
else:
    state_dict = state

# Quitar prefijos 'module.' si los hubiera (va mal si no se hace esto)
new_state_dict = {}
for k, v in state_dict.items():
    if k.startswith("module."):
        new_k = k[len("module."):]
    else:
        new_k = k
    new_state_dict[new_k] = v

model.load_state_dict(new_state_dict, strict=True)
model.eval()

# Guardar el modelo para usarlo (.pt)
out_path = Path("osnet_x1_0_sportsmot.pt")
torch.save(model.state_dict(), out_path)

print(f"Guardado modelo ReID en {out_path.resolve()}")